# **Task I**

- Name : Vaishnavi Agarwal
- Roll No. : 230150028

# Importing Libraries

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.io import wavfile
import shutil
import subprocess
import os
from tqdm import tqdm


# **(a) Create a synthetic time series**

In [9]:
dur = 8.0          # total animation/audio length in seconds
sr_audio = 22050          # audio sampling rate (Hz)
fps = 30                  # animation frames per second
N_frames = int(dur * fps)

t_frames = np.linspace(0, dur, N_frames)

# synthetic time-series:
# trend (slow linear), seasonal (slow sine), higher-frequency tremor, occasional spikes
trend = 0.2 * t_frames # slow drift
seasonal = 1.0 * np.sin(2*np.pi*0.5 * t_frames) # 0.5 Hz wobble
tremor = 0.2 * np.sin(2*np.pi*4.5 * t_frames) # 4.5 Hz small oscillation
np.random.seed(2)
noise = 0.05 * np.random.randn(len(t_frames)) # small noise

spikes = np.zeros_like(t_frames)
spikes_idxs = (np.array([1.5, 3.2, 5.0, 6.6]) / dur * N_frames).astype(int)
for idx in spikes_idxs:
    if 0 <= idx < N_frames:
        spikes[idx: min(idx+3, N_frames)] += 1.5 # short bursts

series = trend + seasonal + tremor + noise + spikes

# normalize series to [-1, +1] for mapping
s_min, s_max = series.min(), series.max()
series_norm = 2*(series - s_min)/(s_max - s_min) - 1.0

# **Animating the sound using time series**

Mapping the time series to pitch of audio and saving the audio

In [10]:

# map series_norm [-1,1] to frequency range [220 Hz, 880 Hz]
f_low = 220.0
f_high = 880.0
frame_freqs = f_low + (series_norm + 1)/2 * (f_high - f_low)  # length N_frames

# synthesize audio 
n_samples = int(dur* sr_audio)
t_audio = np.arange(n_samples) / sr_audio

frame_times = t_frames
inst_freq = np.interp(t_audio, frame_times, frame_freqs)  # in Hz

phase = 0.0
audio = np.zeros(n_samples, dtype=np.float32)
two_pi = 2.0 * np.pi
for n in range(n_samples):
    audio[n] = np.sin(phase)
    phase += two_pi * inst_freq[n] / sr_audio
    if phase > 1e6:
        phase %= two_pi

win_len_samples = int(0.03 * sr_audio)
env = np.ones(n_samples, dtype=np.float32)
env[:win_len_samples] = np.linspace(0.0, 1.0, win_len_samples)
env[-win_len_samples:] = np.linspace(1.0, 0.0, win_len_samples)
audio *= env * 0.9  # scale to avoid clipping

# normalize to int16 for WAV
audio_int16 = np.int16(audio / np.max(np.abs(audio)) * 32767)
wav_name = "time_series_sound.wav"
wavfile.write(wav_name, sr_audio, audio_int16)
print(f"Saved audio file: {wav_name} (sr={sr_audio}, samples={n_samples})")


Saved audio file: time_series_sound.wav (sr=22050, samples=176400)


Building Animations

In [11]:

fig, ax = plt.subplots(figsize=(8,3))
ax.plot(t_frames, series, color='gray', alpha=0.6, label="time series")
line_current, = ax.plot([], [], color='C1', lw=2, label="current window")
cursor, = ax.plot([], [], 'ro', ms=6, label="cursor (time)")
ax.set_xlim(0, dur)
y_margin = 0.1*(series.max()-series.min())
ax.set_ylim(series.min()-y_margin, series.max()+y_margin)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Series value")
ax.legend(loc='upper left')

# for visualizing a short window around cursor
win_display = 0.6  # seconds of data shown as highlighted segment
half_win = win_display/2

def init():
    line_current.set_data([], [])
    cursor.set_data([], [])
    return (line_current, cursor)

def animate(i):
    # i is frame index 0..N_frames-1
    t0 = t_frames[i]
    # find indices within [t0-half_win, t0+half_win]
    left = max(0, t0-half_win)
    right = min(dur, t0+half_win)
    mask = (t_frames >= left) & (t_frames <= right)
    line_current.set_data(t_frames[mask], series[mask])
    cursor.set_data([t0], [series[i]])
    ax.set_title(f"Time {t0:0.2f}s  -> pitch {frame_freqs[i]:0.1f} Hz")
    return (line_current, cursor)

anim = animation.FuncAnimation(
    fig, animate, init_func=init, frames=N_frames, interval=1000/fps, blit=True
)

# Save animation as mp4 (requires ffmpeg installed). Save a version as HTML if not.
video_name = "time_series_animation.mp4"
try:
    print("Saving animation to mp4 (may require ffmpeg)...")
    anim.save(video_name, fps=fps, dpi=150, writer='ffmpeg', codec='h264', bitrate=2000)
    print(f"Saved video: {video_name}")
except Exception as e:
    print("Could not save MP4 with ffmpeg (or ffmpeg not installed). Error:", e)
    # fallback: save as HTML5 video (no audio) or as .gif (large)
    html_name = "time_series_animation.html"
    anim.save(html_name, fps=fps, writer='html', metadata={'artist':'Me'})
    print(f"Saved animation as HTML: {html_name} (open in notebook)")

plt.close(fig)

Saving animation to mp4 (may require ffmpeg)...
Saved video: time_series_animation.mp4


In [ ]:
final_name = "time_series_av.mp4"
ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path and os.path.exists(video_name):
    cmd = [ffmpeg_path, '-y', '-i', video_name, '-i', wav_name, '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', final_name]
    print("Running ffmpeg to mux audio+video...")
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"Saved final muxed file: {final_name}")
    except subprocess.CalledProcessError as e:
        print("ffmpeg muxing failed; output follows:\n", e.stderr.decode('utf-8', errors='ignore'))
else:
    print("ffmpeg not found or video not created. You can mux audio and video later with:")
    print("ffmpeg -y -i time_series_animation.mp4 -i time_series_sound.wav -c:v copy -c:a aac time_series_av.mp4")

# to be checked separately, else audio is not audible.

Running ffmpeg to mux audio+video...
Saved final muxed file: time_series_av.mp4




**Synthetic time series generation**

I made a series composed of:

- a slow linear trend, trend (gives long-term drift),
- a slow seasonal sine at 0.5 Hz, seasonal (captures predictable oscillation),
- a higher-frequency tremor at 4.5 Hz, tremor (adds texture),
- random noise for realism,
- short spikes (bursts) to create salient events.
- Normalized to [-1, 1] so mapping to audio parameters is stable and interpretable.

**Mapping series to pitch**

- I map normalized series to a pitch range of 220–880 Hz (comfortable musical range).
- Rationale: pitch is immediately perceptible to listeners and strongly conveys scalar changes.
- Other options could be amplitude, timbre (add harmonics), or filter cutoff. Pitch was chosen because it’s intuitive.

**Audio synthesis**

- The audio oscillator is a phase-integrator:
- phase[n+1] = phase[n] + 2π * f[n] / sr_audio
- audio sample = sin(phase)

This yields a continuous tone with smoothly varying instantaneous frequency (IF), preserving phase continuity.

Instantaneous frequency array inst_freq is computed by interpolating frame-level frequencies to audio-sample resolution.

**Animation & synchronization**

The animation runs at fps frames per second. We created N_frames = dur * fps frame timestamps.
The audio duration is dur seconds sampled at sr_audio.

By mapping frame timestamps to instantaneous frequencies and interpolating to per-sample frequencies, the audio at time t corresponds to the animation frame at time t — tight synchronization.


# **Why the chosen audio feature is appropriate**

Pitch (frequency) is highly salient to human hearing; mapping series magnitude to pitch makes increases/decreases immediately perceivable. A rising trend raises pitch, local spikes produce sudden pitch jumps, both are easy to notice aurally and complement the visual cursor.

Using pitch rather than amplitude avoids masking small local variations.

# **Challenges**

Frame rate vs audio resolution: audio has far higher temporal resolution. Thus, interpolate frame-level controls to per-sample frequency to avoid stepy sound.

Phase continuity: must integrate phase rather than restarting sine per frame, otherwise audible clicks and unnatural phase discontinuities occur.

ffmpeg availability: best final MP4 if ffmpeg is installed.
